In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import wkt
from shapely.geometry.base import BaseGeometry
import folium
from folium.plugins import MarkerCluster

In [4]:
import sys
sys.path.append("../utils")

import shape_extractor
import maps_api

In [5]:
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [8]:
pd.set_option(
    "display.max_colwidth", 12,
    "display.max_rows", 20,
    "display.max_columns", 50,
    "display.precision", 4
)
plt.style.use('../utils/MyStyle.txt')
cm = 1/2.54
palette = ["#3F3517",  '#CE2E31', '#C96F6B', '#CCA464', '#F8D768', '#F0DCD4']

In [134]:
shapefile_path = "./data/SA1_SHP/SA1_2021_AUST_GDA2020.shp"

API_KEY = ''

PLACES_ENDPOINT = "https://places.googleapis.com/v1/places:searchNearby"

output_excel_name = 'all_places_australia_sa1.xlsx'

# Map

In [6]:
gdf, gdf_m = utils.shape_extractor.final_gdf_creator(shapefile_path)

In [46]:
gdf.tail(4)

,SA1_CODE21,CHG_FLAG21,CHG_LBL21,SA2_CODE21,SA2_NAME21,SA3_CODE21,SA3_NAME21,SA4_CODE21,SA4_NAME21,GCC_CODE21,GCC_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,geometry,centroid,furthest_dist_m
61841,90104100408,0,No change,901041004,Norfolk ...,90104,Norfolk ...,901,Other Te...,9OTER,Other Te...,9,Other Te...,AUS,Australia,2.3801,http://l...,MULTIPOL...,POINT (1...,5609.2759
61842,99797979993,0,No change,997979799,Migrator...,99797,Migrator...,997,Migrator...,99799,Migrator...,9,Other Te...,AUS,Australia,NaN,http://l...,None,None,NaN
61843,99999949999,0,No change,999999499,No usual...,99999,No usual...,999,No usual...,99499,No usual...,9,Other Te...,AUS,Australia,NaN,http://l...,None,None,NaN
61844,ZZZZZZZZZZZ,0,No change,ZZZZZZZZZ,Outside ...,ZZZZZ,Outside ...,ZZZ,Outside ...,ZZZZZ,Outside ...,Z,Outside ...,ZZZ,Outside ...,NaN,http://l...,None,None,NaN


In [40]:
# all_places

In [41]:
# len(all_places)

In [140]:
idx

60789

In [ ]:
# gdf.iloc[idx, :]

In [105]:
origin = None
included_types = [
    # # Health and aged care
    # "chiropractor",
    # "dental_clinic",
    # "dentist",
    # "doctor",
    # "drugstore",
    # "hospital",
    # "massage",
    # "medical_lab",
    # "pharmacy",
    # "physiotherapist",
    # "sauna",
    # "skin_care_clinic",
    # "spa",
    # "tanning_studio",
    # "wellness_center",
    # "yoga_studio",

    # # Education
    # "library",
    # "preschool",
    # "primary_school",
    # "school",
    # "secondary_school",
    # "university",

    # Green, blue and recreation
    "adventure_sports_center",
    "amphitheatre",
    "amusement_center",
    "amusement_park",
    "aquarium",
    "banquet_hall",
    "barbecue_area",
    "botanical_garden",
    "bowling_alley",
    "casino",
    "childrens_camp",
    "comedy_club",
    "community_center",
    "concert_hall",
    "convention_center",
    "cultural_center",
    "cycling_park",
    "dance_hall",
    "dog_park",
    "event_venue",
    "ferris_wheel",
    "garden",
    "hiking_area",
    "historical_landmark",
    "internet_cafe",
    "karaoke",
    "marina",
    "movie_rental",
    "movie_theater",
    "national_park",
    "night_club",
    "observation_deck",
    "off_roading_area",
    "park",
    "philharmonic_hall",
    "picnic_ground",
    "planetarium",
    "plaza",
    "roller_coaster",
    "skateboard_park",
    "state_park",
    "tourist_attraction",
    "video_arcade",
    "visitor_center",
    "water_park",
    "wedding_venue",
    "wildlife_park",
    "wildlife_refuge",
    "zoo",
    "beach",
]

# all_places = []

# for place_type in included_types:
#     print(place_type)
print('Count of retrieved SA1: ', len(all_places))
for idx, row in gdf.iloc[0:, :].iterrows():
    if idx % 100 == 1:
        print(idx, row['SA2_NAME21'])
        print('Count places in last SA1:', len(all_places[-1]))
    if row["centroid"] is not None:
        circle = {
            "circle": {
                "center": {"latitude": row["centroid"].y, "longitude": row["centroid"].x},
                "radius": int(row['furthest_dist_m'])+1
            }
        }

        circle_radius_m = circle['circle']['radius']
        if circle_radius_m > 50000:
            circle_radius_m = 50000
        places = utils.maps_api.search_nearby_places(api_key=API_KEY,
                                                places_end_point=PLACES_ENDPOINT,
                                                included_types=included_types,
                                                page_size=20,
                                                circle_center=tuple(circle['circle']['center'].values()),
                                                circle_radius_m=circle_radius_m,
                                                routing_origin=origin,
                                                travel_mode="DRIVE"
                                                )

        places_df = pd.json_normalize(places)

        if places_df is not None and not places_df.empty:
            # places_df['place_search_type'] = place_type
            places_df["SA1_CODE21"] = row["SA1_CODE21"]
            places_df["SA2_CODE21"] = row["SA2_CODE21"]
            places_df["SA2_NAME21"] = row["SA2_NAME21"]
            places_df["SA3_NAME21"] = row['SA3_NAME21'] 
            places_df["SA3_CODE21"] = row['SA3_CODE21'] 
            places_df["STE_NAME21"] = row['STE_NAME21'] 
            all_places.append(places_df)

In [141]:
included_types2 = [
    # Green, blue, and recreation
    "arena",
    "athletic_field",
    "fishing_charter",
    "fishing_pond",
    "fitness_center",
    "golf_course",
    "gym",
    "ice_skating_rink",
    "playground",
    "ski_resort",
    "sports_activity_location",
    "sports_club",
    "sports_coaching",
    "sports_complex",
    "stadium",
    "swimming_pool",

    # Arts and culture
    "art_gallery",
    "art_studio",
    "auditorium",
    "cultural_landmark",
    "historical_place",
    "monument",
    "museum",
    "performing_arts_theater",
    "sculpture",
    "opera_house",
    "library",

    # Religion
    "church",
    "hindu_temple",
    "mosque",
    "synagogue",

    # Justice and emergency services
    "fire_station",
    "police",
    "post_office",
    "courthouse",
]

# for place_type in included_types:
#     print(place_type)
print('Count of retrieved SA1: ', len(all_places))
for idx, row in gdf.iloc[60789:, :].iterrows():
    if idx % 100 == 1:
        print(idx, row['SA2_NAME21'])
        print('Count places in last SA1:', len(all_places[-1]))
    if row["centroid"] is not None:
        circle = {
            "circle": {
                "center": {"latitude": row["centroid"].y, "longitude": row["centroid"].x},
                "radius": int(row['furthest_dist_m'])+1
            }
        }

        circle_radius_m = circle['circle']['radius']
        if circle_radius_m > 50000:
            circle_radius_m = 50000
        places = utils.maps_api.search_nearby_places(api_key=API_KEY,
                                                places_end_point=PLACES_ENDPOINT,
                                                included_types=included_types2,
                                                page_size=20,
                                                circle_center=tuple(circle['circle']['center'].values()),
                                                circle_radius_m=circle_radius_m,
                                                routing_origin=origin,
                                                travel_mode="DRIVE"
                                                )

        places_df = pd.json_normalize(places)

        if places_df is not None and not places_df.empty:
            # places_df['place_search_type'] = place_type
            places_df["SA1_CODE21"] = row["SA1_CODE21"]
            places_df["SA2_CODE21"] = row["SA2_CODE21"]
            places_df["SA2_NAME21"] = row["SA2_NAME21"]
            places_df["SA3_NAME21"] = row['SA3_NAME21'] 
            places_df["SA3_CODE21"] = row['SA3_CODE21'] 
            places_df["STE_NAME21"] = row['STE_NAME21'] 
            all_places.append(places_df)

Count of retrieved SA1:  105198
60801 Macgregor (ACT)
Count places in last SA1: 3
60901 Bonner
Count places in last SA1: 1
61001 Gungahlin
Count places in last SA1: 19
61101 Palmerston
Count places in last SA1: 2
61201 Downer
Count places in last SA1: 9
61301 Griffith (ACT)
Count places in last SA1: 10
61401 Calwell
Count places in last SA1: 1
61501 Greenway
Count places in last SA1: 3
61601 Theodore
Count places in last SA1: 1
61701 Curtin
Count places in last SA1: 2
61801 Molonglo
Count places in last SA1: 20


In [142]:
all_places_df = pd.concat(all_places, ignore_index=True)
all_places_df.tail()

,id,types,rating,userRatingCount,location.latitude,location.longitude,displayName.text,displayName.languageCode,accessibilityOptions.wheelchairAccessibleParking,accessibilityOptions.wheelchairAccessibleEntrance,regularOpeningHours.openNow,regularOpeningHours.periods,regularOpeningHours.weekdayDescriptions,regularOpeningHours.nextOpenTime,parkingOptions.freeParkingLot,accessibilityOptions.wheelchairAccessibleRestroom,accessibilityOptions.wheelchairAccessibleSeating,addressDescriptor.landmarks,restroom,addressDescriptor.areas,SA1_CODE21,SA2_CODE21,SA2_NAME21,SA3_NAME21,SA3_CODE21,STE_NAME21,regularOpeningHours.nextCloseTime,parkingOptions.freeStreetParking,priceRange.startPrice.currencyCode,priceRange.startPrice.units,priceRange.endPrice.currencyCode,priceRange.endPrice.units,priceLevel,parkingOptions.paidParkingLot,parkingOptions.paidGarageParking,parkingOptions.freeGarageParking,parkingOptions.paidStreetParking,parkingOptions.valetParking
708479,ChIJa7Vs...,"[museum,...",4.8,4.0,-29.0555,167.9574,Commissa...,en,NaN,NaN,False,[{'open'...,[Monday:...,2025-09-...,NaN,NaN,NaN,[{'name'...,NaN,[{'name'...,90104100407,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
708480,ChIJVwck...,"[museum,...",4.6,7.0,-29.0262,167.9405,Discover...,en,True,True,False,[{'open'...,[Monday:...,2025-09-...,NaN,True,NaN,[{'name'...,True,[{'name'...,90104100407,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
708481,ChIJxX_M...,[histori...,4.3,3.0,-29.0502,167.9531,Watermil...,en,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[{'name'...,NaN,NaN,90104100407,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
708482,ChIJbQRb...,"[gym, fi...",5.0,2.0,-29.0343,167.9553,Norfolk ...,en,True,NaN,False,[{'open'...,[Monday:...,2025-09-...,NaN,NaN,NaN,[{'name'...,True,NaN,90104100407,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
708483,ChIJVVUl...,"[police,...",NaN,NaN,-29.0311,167.9577,Police S...,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[{'name'...,NaN,[{'name'...,90104100407,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [143]:
all_places_df_unique = all_places_df.drop_duplicates(subset='id').reset_index().copy()
all_places_df_unique.tail()

,index,id,types,rating,userRatingCount,location.latitude,location.longitude,displayName.text,displayName.languageCode,accessibilityOptions.wheelchairAccessibleParking,accessibilityOptions.wheelchairAccessibleEntrance,regularOpeningHours.openNow,regularOpeningHours.periods,regularOpeningHours.weekdayDescriptions,regularOpeningHours.nextOpenTime,parkingOptions.freeParkingLot,accessibilityOptions.wheelchairAccessibleRestroom,accessibilityOptions.wheelchairAccessibleSeating,addressDescriptor.landmarks,restroom,addressDescriptor.areas,SA1_CODE21,SA2_CODE21,SA2_NAME21,SA3_NAME21,SA3_CODE21,STE_NAME21,regularOpeningHours.nextCloseTime,parkingOptions.freeStreetParking,priceRange.startPrice.currencyCode,priceRange.startPrice.units,priceRange.endPrice.currencyCode,priceRange.endPrice.units,priceLevel,parkingOptions.paidParkingLot,parkingOptions.paidGarageParking,parkingOptions.freeGarageParking,parkingOptions.paidStreetParking,parkingOptions.valetParking
247408,708454,ChIJnxyk...,[sports_...,NaN,NaN,-29.0334,167.9541,Norfolk ...,en,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[{'name'...,True,[{'name'...,90104100404,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
247409,708455,ChIJT-Bq...,[athleti...,5.0,1.0,-29.0330,167.9458,Norfolk ...,en,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[{'name'...,NaN,[{'name'...,90104100404,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
247410,708456,ChIJAQDA...,[sports_...,NaN,NaN,-29.0332,167.9461,Squash C...,en,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[{'name'...,NaN,[{'name'...,90104100404,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
247411,708458,ChIJA3zK...,[fire_st...,NaN,NaN,-29.0356,167.9449,Norfolk ...,en,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[{'name'...,True,[{'name'...,90104100405,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
247412,708472,ChIJsRd8...,[sports_...,4.8,6.0,-29.0070,167.9201,Norfolk ...,en,NaN,NaN,False,[{'open'...,[Monday:...,2025-09-...,NaN,NaN,NaN,NaN,NaN,NaN,90104100407,901041004,Norfolk ...,Norfolk ...,90104,Other Te...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [144]:
sum(all_places_df_unique.userRatingCount.dropna())

15903062.0

In [145]:
gdf_places = gpd.GeoDataFrame(
    all_places_df_unique,
    geometry=gpd.points_from_xy(all_places_df_unique["location.longitude"], all_places_df_unique["location.latitude"]),
    crs=gdf.crs
)
joined = gpd.sjoin(gdf_places, gdf, how="left", predicate="within")
joined["SA3_CODE21_right"] = joined["SA3_CODE21_right"].fillna(joined["SA3_CODE21_left"])
joined["SA3_NAME21_right"] = joined["SA3_NAME21_right"].fillna(joined["SA3_NAME21_left"])

In [146]:
len(joined)

247413

In [147]:
joined.to_excel('Raw_' + output_excel_name)

In [148]:
cols_to_drop = [
    "regularOpeningHours.openNow",
    "regularOpeningHours.nextCloseTime",
    "regularOpeningHours.nextOpenTime",
    "displayName.languageCode",
    "addressDescriptor.landmarks",
    "addressDescriptor.areas",
    "priceRange.startPrice.currencyCode",
    "priceRange.endPrice.currencyCode",
]

joined = joined.drop(columns=[c for c in cols_to_drop if c in joined.columns])

In [149]:
rename_map = {
    "location.latitude": "latitude",
    "location.longitude": "longitude",
    "regularOpeningHours.periods": "opening_periods",
    "regularOpeningHours.weekdayDescriptions": "opening_weekdays",
    "displayName.text": "place_name",
    "accessibilityOptions.wheelchairAccessibleParking": "wheelchair_parking",
    "accessibilityOptions.wheelchairAccessibleEntrance": "wheelchair_entrance",
    "accessibilityOptions.wheelchairAccessibleRestroom": "wheelchair_restroom",
    "parkingOptions.freeParkingLot": "free_parking_lot",
    "parkingOptions.paidParkingLot": "paid_parking_lot",
    "priceLevel": "price_level",
    "accessibilityOptions.wheelchairAccessibleSeating": "wheelchair_seating",
    "parkingOptions.paidStreetParking": "paid_street_parking",
    "parkingOptions.paidGarageParking": "paid_garage_parking",
    "parkingOptions.freeStreetParking": "free_street_parking",
    "priceRange.startPrice.units": "start_price_units",
    "priceRange.endPrice.units": "end_price_units",
    "SA3_CODE21_right": "SA3_CODE21",
    "SA3_NAME21_right": "SA3_NAME21",
    "CHG_FLAG21": "change_flag",
    "CHG_LBL21": "change_label",
    "SA4_CODE21": "SA4_CODE21",
    "SA4_NAME21": "SA4_NAME21",
    "GCC_CODE21": "GCC_CODE21",
    "GCC_NAME21": "GCC_NAME21",
    "STE_CODE21": "state_code",
    "STE_NAME21_right": "state_name",
    "AUS_CODE21": "AUS_CODE21",
    "AUS_NAME21": "AUS_NAME21",
    "AREASQKM21": "area_sqkm",
    "LOCI_URI21": "loci_uri",
    "centroid": "geometry_centroid",
    "furthest_dist_m": "furthest_distance_m",
}

joined = joined.rename(columns=rename_map)

In [150]:
joined.to_excel(output_excel_name)

# Folium Map

In [9]:
df = pd.read_excel('../data/places/all_places_australia_sa1.xlsx')
df['latitude'] = pd.to_numeric(df['latitude'], errors="coerce")
df['longitude'] = pd.to_numeric(df['longitude'], errors="coerce")
df = df.dropna(subset=['latitude', 'longitude'])

In [10]:
def style_fn(_feature):
    return {
        "fillOpacity": 0.0,
        "color": "#555555",
        "weight": 1
    }

def highlight_fn(_feature):
    return {
        "fillOpacity": 0.1,
        "color": "#000000",
        "weight": 2
    }

In [17]:
center = [df['latitude'].mean(), df['longitude'].mean()]
m = folium.Map(location=center, zoom_start=10, tiles="OpenStreetMap")

cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    popup = (str(row['place_name']) if 'place_name' else f"{row['latitude']:.6f}, {row['longitude']:.6f}",
            str('Rating: ') + str(row['rating']) if 'rating' else f"{row['latitude']:.6f}, {row['longitude']:.6f}",
            str('RatingCount: ') + str(row['userRatingCount']) if 'userRatingCount' else f"{row['latitude']:.6f}, {row['longitude']:.6f}",
            )
    tooltip = popup
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=popup,
        tooltip=tooltip,
    ).add_to(cluster)

# sa3 = gdf.to_crs(epsg=4326)

# folium.GeoJson(
#     sa3['geometry'].to_json(),
#     name="SA3 Boundaries",
#     style_function=style_fn,
#     highlight_function=highlight_fn,
#     tooltip=folium.GeoJsonTooltip(
#         fields=[c for c in ["SA3_NAME21", "SA3_CODE21", "SA4_NAME21", "state_name"] if c in sa3.columns],
#         aliases=["SA3", "SA3 Code", "SA4", "State"],
#         sticky=False
#     ),
# ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save("../data/places/points_map_sa1.html")
print("Saved to points_map.html")

Saved to points_map.html
